# GRU-joint short-horizon prediction từ Robot EE

Notebook này train **một GRU chung** nhận `[x, y, z, Δx, Δy, Δz]` và dự đoán `[x, y, z]` sau 5 bước. Cấu hình mặc định: resample 15 Hz, window 20, horizon 5 (~0.333 s), chỉ dùng `GROUND_TRUTH`, loại trial lỗi `Long/experiment_GROUND_TRUTH_20260616_160434.csv`.

## Cách chạy

1. Bundle `gru_robot_ee_colab_bundle.zip` đã được chuẩn bị trong workspace.
2. Upload notebook này lên Google Colab.
3. Chọn **Runtime → Change runtime type → T4 GPU**.
4. Chạy lần lượt các cell; khi được hỏi, upload bundle ZIP.
5. Cell train có thể chạy lâu, không cần giữ kết nối với Codex.
6. Cell cuối kiểm tra model và tải artifact ZIP về máy.

> Notebook không thay model SVGP hoặc cấu hình ROS. Việc tích hợp launch được thực hiện riêng sau khi đánh giá model.

In [ ]:
import json, os, shutil, subprocess, sys, time, zipfile
from datetime import datetime
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if not IN_COLAB:
    print('Bạn đang chạy local. Để train bằng GPU, hãy mở notebook này trong Google Colab.')

import tensorflow as tf
print('Python:', sys.version.split()[0])
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
if IN_COLAB and not tf.config.list_physical_devices('GPU'):
    raise RuntimeError('Colab chưa có GPU. Chọn Runtime → Change runtime type → T4 GPU rồi chạy lại.')

## Tùy chọn: tạo lại bundle trên máy local

Không cần chạy cell dưới đây nếu bundle đã có. Nếu dataset hoặc script train thay đổi, mở notebook bằng Jupyter local và đổi `BUNDLE_VERSION` để tạo bundle mới. Cell không ghi đè bundle cũ.

In [ ]:
BUNDLE_VERSION = 'v1'
LOCAL_WORKSPACE = Path('/home/hungnb/cocarry_ws')
LOCAL_DATA_DIR = Path('/home/hungnb/simulation_hri/cocarry_logs')
LOCAL_TRAIN_SCRIPT = LOCAL_WORKSPACE / 'train_gru_robot_ee.py'
LOCAL_BUNDLE = LOCAL_WORKSPACE / 'gru_robot_ee_colab_bundle.zip'
EXCLUDED = {'Long/experiment_GROUND_TRUTH_20260616_160434.csv'}

if IN_COLAB:
    print('Bỏ qua cell đóng gói local trên Colab.')
elif LOCAL_BUNDLE.exists():
    print('Bundle đã tồn tại, không ghi đè:', LOCAL_BUNDLE)
else:
    if not LOCAL_TRAIN_SCRIPT.is_file() or not LOCAL_DATA_DIR.is_dir():
        raise FileNotFoundError('Không tìm thấy script train hoặc cocarry_logs trên máy local.')
    filtered = {}
    for split in ('train', 'val', 'test'):
        entries = json.loads((LOCAL_DATA_DIR / f'{split}_file_list.json').read_text(encoding='utf-8'))
        filtered[split] = [p for p in entries if 'GROUND_TRUTH' in Path(p).name.upper() and p not in EXCLUDED]
    with zipfile.ZipFile(LOCAL_BUNDLE, 'x', zipfile.ZIP_DEFLATED) as archive:
        archive.write(LOCAL_TRAIN_SCRIPT, 'train_gru_robot_ee.py')
        for split, entries in filtered.items():
            archive.writestr(f'cocarry_logs/{split}_file_list.json', json.dumps(entries, ensure_ascii=False, indent=2) + '\n')
            for relative_name in entries:
                archive.write(LOCAL_DATA_DIR / relative_name, f'cocarry_logs/{relative_name}')
        archive.writestr('bundle_metadata.json', json.dumps({
            'version': BUNDLE_VERSION, 'ground_truth_only': True,
            'excluded_files': sorted(EXCLUDED),
            'split_counts': {k: len(v) for k, v in filtered.items()}
        }, ensure_ascii=False, indent=2) + '\n')
    print('Đã tạo:', LOCAL_BUNDLE, f'({LOCAL_BUNDLE.stat().st_size / 2**20:.1f} MiB)')

## Upload và kiểm tra bundle trên Colab

Chọn file `gru_robot_ee_colab_bundle.zip` từ workspace. Mỗi lần chạy tạo thư mục giải nén mới, không ghi đè dữ liệu cũ.

In [ ]:
if not IN_COLAB:
    raise RuntimeError('Cell này dành cho Google Colab.')
from google.colab import files
uploaded = files.upload()
zip_names = [name for name in uploaded if name.endswith('.zip')]
if len(zip_names) != 1:
    raise ValueError('Hãy upload đúng một file bundle ZIP.')

BUNDLE_ZIP = Path('/content') / zip_names[0]
EXTRACT_ROOT = Path('/content') / f"gru_robot_ee_bundle_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
EXTRACT_ROOT.mkdir(parents=True, exist_ok=False)
with zipfile.ZipFile(BUNDLE_ZIP, 'r') as archive:
    archive.extractall(EXTRACT_ROOT)
DATA_DIR = EXTRACT_ROOT / 'cocarry_logs'
TRAIN_SCRIPT = EXTRACT_ROOT / 'train_gru_robot_ee.py'
bundle_meta = json.loads((EXTRACT_ROOT / 'bundle_metadata.json').read_text(encoding='utf-8'))
print('Bundle:', bundle_meta)
print('Data:', DATA_DIR)
print('Train script:', TRAIN_SCRIPT)
assert TRAIN_SCRIPT.is_file()
assert all((DATA_DIR / f'{s}_file_list.json').is_file() for s in ('train', 'val', 'test'))

## Cấu hình train

Các giá trị dưới đây là cấu hình đã chốt. Có thể giảm `EPOCHS` hoặc chỉ giữ một seed để chạy thử nhanh; lượt train chính nên giữ nguyên.

In [ ]:
SAMPLE_RATE = 15.0
WINDOW_SIZE = 20
HORIZON = 5
LAYERS = '1,2,3'
UNITS = '32,64'
SEEDS = '42,43,44'
DROPOUT = 0.2
LEARNING_RATE = 0.001
BATCH_SIZE = 128
EPOCHS = 200
PATIENCE = 20
LR_PATIENCE = 10

RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_DIR = Path('/content') / f'gru_robot_ee_joint_w{WINDOW_SIZE}_h{HORIZON}_{RUN_ID}'
print('Output mới:', OUTPUT_DIR)

## Train

Cell này chạy ablation GRU-joint và có thể mất thời gian. Model được chọn bằng trung bình validation qua ba seed; tập test chỉ được dùng sau khi đã chọn cấu hình. Không có so sánh với SVGP.

In [ ]:
command = [
    sys.executable, str(TRAIN_SCRIPT),
    '--data-dir', str(DATA_DIR), '--output-dir', str(OUTPUT_DIR),
    '--sample-rate', str(SAMPLE_RATE), '--window-size', str(WINDOW_SIZE),
    '--horizon', str(HORIZON), '--layers', LAYERS, '--units', UNITS,
    '--seeds', SEEDS, '--dropout', str(DROPOUT),
    '--learning-rate', str(LEARNING_RATE), '--batch-size', str(BATCH_SIZE),
    '--epochs', str(EPOCHS), '--patience', str(PATIENCE),
    '--lr-patience', str(LR_PATIENCE),
]
print(' '.join(command))
subprocess.run(command, check=True)

## Xem kết quả

Bảng đầu tiên cho biết validation trung bình của từng cấu hình. Các số test chỉ thuộc model đã được khóa sau bước chọn validation.

In [ ]:
import pandas as pd
metadata = json.loads((OUTPUT_DIR / 'metadata.json').read_text(encoding='utf-8'))
rows = []
for name, error_m in metadata['search']['mean_validation_error_by_config_m'].items():
    rows.append({'configuration': name, 'mean_validation_3d_mm': error_m * 1000.0})
display(pd.DataFrame(rows).sort_values('mean_validation_3d_mm').reset_index(drop=True))
print('Selected run:', metadata['selected_run'])
print('Validation metrics:', metadata['validation_metrics'])
print('Test metrics:', metadata['test_metrics'])
print('Batch-1 inference mean:', metadata['batch1_test_inference_ms_mean'], 'ms')

## Kiểm tra và tải artifact

Cell cuối reload model, chạy một inference giả để kiểm tra shape/NaN, sau đó tạo ZIP gồm model, scaler, metadata, search results và ba manifest đã lọc.

In [ ]:
import numpy as np
model_path = OUTPUT_DIR / metadata['model_file']
loaded_model = tf.keras.models.load_model(model_path, compile=False)
dummy = np.zeros((1, WINDOW_SIZE, 6), dtype=np.float32)
dummy_prediction = loaded_model.predict_on_batch(dummy)
if hasattr(dummy_prediction, 'numpy'):
    dummy_prediction = dummy_prediction.numpy()
assert dummy_prediction.shape == (1, 3)
assert np.isfinite(dummy_prediction).all()
print('Reload/inference OK:', dummy_prediction)

ARTIFACT_ZIP = Path('/content') / f'{OUTPUT_DIR.name}.zip'
if ARTIFACT_ZIP.exists():
    raise FileExistsError(f'Từ chối ghi đè {ARTIFACT_ZIP}')
with zipfile.ZipFile(ARTIFACT_ZIP, 'x', zipfile.ZIP_DEFLATED) as archive:
    for path in OUTPUT_DIR.rglob('*'):
        if path.is_file():
            archive.write(path, Path(OUTPUT_DIR.name) / path.relative_to(OUTPUT_DIR))
print('Artifact:', ARTIFACT_ZIP, f'({ARTIFACT_ZIP.stat().st_size / 2**20:.1f} MiB)')
files.download(str(ARTIFACT_ZIP))

## Sau khi train

Không chép đè thư mục SVGP. Hãy giữ nguyên ZIP và gửi `metadata.json` cùng kết quả test để đánh giá trước. Khi tích hợp ROS, GRU này yêu cầu `window_size=20`, `num_features=6`, 15 Hz và velocity đúng bằng `Δp` mỗi mẫu; runtime cũ có phép chia cố định 16 Hz nên phải sửa/kiểm thử riêng trước simulation.